In [1]:
import polars as pl
import pandas as pd
import numpy as np
import datetime as dt
import os
import sys
import json 
import importlib

sys.path.append("../utils/")

import helpers as hp

In [2]:
rng = np.random.default_rng(seed=274)

In [3]:

with open("../configs/zone_distances.json", "r") as json_file:
    zone_distances = json.load(json_file)

In [4]:
zone_distances

{'Yerbabuena': {'centroids': [20.964404421308334, -101.2847459818324],
  'points': [{'point': [20.9743, -101.300626],
    'distance': 0.01871089133756122},
   {'point': [20.946278, -101.298483], 'distance': 0.022743632462390598},
   {'point': [20.951706, -101.264939], 'distance': 0.023527992541503697},
   {'point': [20.979447, -101.274624], 'distance': 0.018131014585796808},
   {'point': [20.985938, -101.285636], 'distance': 0.02155196379935847},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122}]},
 'Marfil': {'centroids': [20.998877268148007, -101.29015919031652],
  'points': [{'point': [20.979212, -101.292779],
    'distance': 0.019839006379117525},
   {'point': [20.991242, -101.304647], 'distance': 0.016376628136369357},
   {'point': [21.006043, -101.294156], 'distance': 0.008205010702043177},
   {'point': [21.019272, -101.283363], 'distance': 0.021497285645706604},
   {'point': [21.008196, -101.275124], 'distance': 0.017688858391175503},
   {'point': [20.999863,

In [5]:
gym_hours = {"open" : 6,
             "close" : 22}

profiles = {
"1": {
"name" : "frequent_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .89},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .76},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .85},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .7},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.9},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .9},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 6.43,
"hour_std" : .75,
"plan_probability" : {"1": 0.05, "2": 0.2, "3" : 0.45, "4": 0.3}
},
"2" : {
"name" : "frequent_noon",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .7},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .82},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .67},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .87},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.78},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .84},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 13.16,
"hour_std" : 1.40,
"plan_probability" : {"1": .2, "2": 0.5, "3" : 0.25, "4": 0.05}
            },
"3" :{
"name" : "frequent_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .74},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .87},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .76},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .84},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 19.5,
"hour_std" : 1.15,
"plan_probability" : {"1": 0.15, "2": 0.15, "3" : 0.45, "4": 0.25}
},
"4" :{
"name" : "random_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .4},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .4},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .3},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 0.5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" : .6},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : 1},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 8.5,
"hour_std" : 2.05,
"plan_probability" : {"1": 0.65, "2": 0.25, "3" : 0.1, "4": 0}
},
"5" :{
"name" : "random_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .3},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .3},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .6},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.9},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 20.1,
"hour_std" : 2.08,
"plan_probability" : {"1": 0.4, "2": 0.3, "3" : 0.2, "4": 0.1}
},
"6" :{
"name" : "full_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .5},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .5},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.5},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .5},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 14.00,
"hour_std" : 3.0,
"plan_probability" : {"1": 0.59, "2": 0.3, "3" : 0.1, "4": 0.01}
},
"7" :{
"name" : "rare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .2},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .4},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .2},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.3},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .4},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 16.0,
"hour_std" : 3.5,
"plan_probability" : {"1": 0.85, "2": 0.1, "3" : 0.05, "4": 0}
},
"8" :{
"name" : "ultrarare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .1},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .1},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.2},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : 14.5,
"hour_std" : 5.5,
"plan_probability" : {"1": 0.9, "2": 0.08, "3" : 0.02, "4": 0}
}
}

profile_weights = {"1" : 0.23076923, "2": 0.12820513, "3" : 0.25641026, "4" : 0.07692308, "5" : 0.07692308, "6" : 0.05128205, "7" : 0.07692308, "8" : 0.1025641}

for profile in profiles.keys():
    print(profile)
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
        
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    
    profiles[profile]["total_weight"] = sum_weight

    assert 1 == sum([i for i in profiles[profile]["plan_probability"].values() ])
profiles

1
2
3
4
5
6
7
8


{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.89,
    'day_probability': 0.178},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.76,
    'day_probability': 0.152},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.85,
    'day_probability': 0.16999999999999998},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.7,
    'day_probability': 0.13999999999999999},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': 6.43,
  'hour_std': 0.75,
  'plan_probability': {'1': 0.05, '2': 0.2, '3': 0.45, '4': 0.3},
  'total_weight': 5.0},
 '2': {'name': 'frequent

In [6]:
# Functions
def create_customer_profiles(profile_weights_ : dict, n_):
    return {str(i): str(int(rng.choice( list(profile_weights.keys()), p = list(profile_weights.values()) ) ) ) for i in range(n_) }

In [7]:
create_customer_profiles(profiles, 9)

{'0': '8',
 '1': '1',
 '2': '3',
 '3': '7',
 '4': '3',
 '5': '4',
 '6': '1',
 '7': '5',
 '8': '1'}

In [8]:
# prob = np.array((.9,0.5,1,.3,.3,.2,.3,.4))

# prob/sum(prob)

In [9]:
importlib.reload(hp)

oli =hp.create_customer(zone_distances)

Creating customer
36254


In [10]:
oli

{'id': None,
 'name': 'Natividad Eva',
 'last_name': 'Montaño Nazario',
 'gender': np.False_,
 'birth_date': datetime.datetime(1985, 9, 7, 0, 0),
 'lat': 20.98821795202615,
 'lon': -101.29917077337164,
 'zipcode': 36254,
 'email': 'q***************w@gmail.com',
 'phone_number': '(473)6043690',
 'created_at': datetime.datetime(2026, 6, 1, 21, 36, 27, 105128),
 'status': 'Active',
 'profile_type': None,
 'updated_at': datetime.datetime(2026, 6, 1, 21, 36, 27, 105134)}

In [11]:
def create_customer_access_data(date_, profile_metadata_, gym_hours_, data_path_ = "../data/", customers_file_ = "customers.csv", payment_file_ = "payments.csv", access_file_ = "access.csv", plans_file_ = "suscription_plans.csv"):
    
    opening_date = "2026-01-02"

    assert date_ >= opening_date, f"Gym doesn't exists before {opening_date}"

    # Define Schemas
    customers_schema = {"Customer_Id" :pl.Int64,
                        "Name": pl.String,
                        "Last_Name": pl.String,
                        "Gender" : pl.Boolean,
                        "Birth_Date" : pl.Datetime,
                        "Latitude": pl.Float32,
                        "Longitude" : pl.Float32,
                        "Zipcode" : pl.Int64 ,
                        "Email" : pl.String,
                        "Phone_Number" : pl.String,
                        "Created_At" : pl.Date,
                        "Status" : pl.String,
                        "Profile_Type" : pl.String,
                        "Updated_At" : pl.Datetime}
    

    payments_schema = {"Payment_Id" : pl.String,
                       "Plan_Id" : pl.Int64,
                       "Customer_Id" : pl.Int64,
                       "Payment_Amount" : pl.Float16,
                       "Payment_Time" : pl.Datetime,
                       "Plan_Expiration_Time" : pl.Datetime,
                       "Payment_Status": pl.String,
                       "Updated_At" : pl.Datetime
    }

    access_schema = {"Visit_Id" : pl.Int64,
                     "Customer_Id" : pl.Int64,
                     "Plan_Id" : pl.Int64,
                     "Branch_Id" : pl.Int8,
                     "Payment_Id" : pl.String,
                     "Date" : pl.Date,
                     "Visit_Time" : pl.Datetime,
                     "Updated_At" : pl.Datetime
    }

    suscription_plan_schema = {"Plan_Id" : pl.Int64,
                                "Plan_Name" : pl.String,
                                "Plan_Cost" : pl.Float16, 
                                "Plan_Duration" : pl.Int64,
                                "Plan_Start_Time" : pl.Datetime, 
                                "Plan_End_Time" : pl.Datetime, 
                                "Plan_Description" :pl.String,
                                "Updated_At": pl.Datetime
                                }
    
    eval_date = dt.datetime.strptime(date_, "%Y-%m-%d")
    day_names_dic = {0 : 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4:'Friday', 5:'Saturday', 6: 'Sunday'}
    day_of_week = eval_date.weekday()
    day_name = day_names_dic[day_of_week]

    print(eval_date, day_of_week, day_name)
    
    if not os.path.isdir(data_path_):
        os.makedirs(data_path_, exist_ok = True)

    if not os.path.exists(data_path_ + customers_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = customers_schema,
                                                    orient="row").write_csv(data_path_+ customers_file_)
    
    if not os.path.exists(data_path_ + payment_file_):
        print("Creating payments file")
        df_payments = pl.DataFrame([], schema = payments_schema,
                                                    orient="row").write_csv(data_path_+ payment_file_)
        
    if not os.path.exists(data_path_ + access_file_):
        print("Creating access registry file")
        df_access = pl.DataFrame([], schema = access_schema,
                                                    orient="row").write_csv(data_path_+ access_file_)

    df_customers = pl.read_csv(data_path_+ customers_file_, schema=customers_schema)
    df_access = pl.read_csv(data_path_+ access_file_, schema=access_schema)
    df_plans = pl.read_csv(data_path_ + plans_file_, schema = suscription_plan_schema)
    df_payments = pl.read_csv(data_path_+ payment_file_, schema=payments_schema)
    
    #########################
    # 1. Create new customers
    #########################

    # 1.1. Customers configs
    tot_new = int(abs(rng.normal(0, 1)))

    max_id = df_customers.select(pl.max("Customer_Id")).item()

    print(max_id)
    counter = max_id
    if max_id == None:
        counter = 0
        tot_new = 12

    print("Total new customers", tot_new)
    counter += 1 

    dict_profiles = create_customer_profiles(profile_metadata_, tot_new)

    # 1.2 Generate customers
    new_customers = []
    new_customers_ids = []
    for new_cus in range(tot_new):

        new_customer = hp.create_customer(zone_distances)
        new_customer["id"] = counter
        new_customer["profile_type"] = dict_profiles[str(new_cus)]
        new_customer["created_at"] = dt.datetime.strptime(date_, "%Y-%m-%d").date()

        new_customers.append(tuple(new_customer.values()))
        new_customers_ids.append(counter)
        counter +=1

    # 1.3 Save file
    df_new_customers = pl.DataFrame(new_customers, customers_schema, orient = "row")
    df_customers = pl.concat([df_customers, df_new_customers])
    df_customers.write_csv(data_path_ + customers_file_)

    ###########################
    # 2. Create Access Registry
    ###########################

    # 2.1 Given at least one customer, generate random visits
    if df_customers.shape[0] >0:
        day_cus = df_customers.filter(pl.col("Status") =="Active").to_dicts()

        
        payments = []
        access = []
        
        access_count = df_access.select(pl.max("Visit_Id")).item()
        if access_count == None:
            access_count = 0

        for row in day_cus:
            cus_id = row["Customer_Id"]
            profile = str(row["Profile_Type"])
            profile_data = profile_metadata_[profile]
            day_proba  = profile_data["days_probability"][day_name[0:3].lower()]["day_probability"]

            has_visit = rng.choice([1,0], p = [day_proba, 1-day_proba])

            if max_id == None:
                has_visit = 1

            customer_new = "old customer"
            if cus_id in new_customers_ids: # If it's a new customer, has to enter gym
                has_visit = 1
                customer_new = "new_customer"

            print(f"({customer_new}) Customer {cus_id} with profile {profile} appeared {bool(has_visit)}")

            # Check is customer assisted
            if has_visit == 0:
                continue
            
            ######################
            # 2.2. Create Payments
            ######################

            #Check if plan is still active
            temp_pay = df_payments.filter(pl.col("Customer_Id") == cus_id).sort(by = ["Plan_Expiration_Time"], descending=[True])
            #display(temp_pay)
            payment_count = temp_pay.shape[0]

            
            expiration_date = temp_pay.select(pl.max("Plan_Expiration_Time")).item()

            while True:
                arrival_time = rng.normal(profile_data["hour_mean"], profile_data["hour_std"])
                if (arrival_time >= gym_hours_['open']) and (arrival_time < gym_hours_["close"]-1):
                    break
            
            arrival_hour = int(arrival_time)
            arrival_minute = int(arrival_time % arrival_hour * 60)
            
            arrival_time_obj = dt.time(arrival_hour, arrival_minute, rng.integers(0,59))

            plan_choice = 0
            #display(temp_pay.head(1))
            
            last_pay = temp_pay.head(1)
            if last_pay.is_empty() == False:
                plan_choice = last_pay.select(pl.col("Plan_Id")).item()
                print("checando plan_choice", plan_choice)

            momentum_pass_visits = 0
            if plan_choice == 2:
                payment_id = df_access.filter(pl.col("Customer_Id") == cus_id).sort(by = ["Visit_Id"], descending=[True]).head(1).select(pl.col("Payment_Id")).item()
                momentum_pass_visits += df_access.filter(pl.col("Payment_Id") == payment_id).shape[0]
            # Generate payment if conditions are meet

            print(payment_count, expiration_date, momentum_pass_visits)
            if (payment_count == 0) or (expiration_date < dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj)) or (momentum_pass_visits >= 5) :
                
                plan_choice = int(rng.choice( list(profile_metadata_[profile]["plan_probability"].keys()), p= list(profile_metadata_[profile]["plan_probability"].values())))

                if cus_id in new_customers_ids: # If it's a new customer, has to enter gym
                    plan_choice = 1

                print(f"Customer {cus_id} is paying plan {plan_choice}")

                temp_plan  = df_plans.filter(pl.col("Plan_Id")== plan_choice)

                amount = temp_plan.select(pl.col("Plan_Cost")).item()
                duration = temp_plan.select(pl.col("Plan_Duration")).item()
                payment_count +=1
                payments_dict = {"Payment_Id" : str(cus_id) + "-" + str(payment_count),
                                "Plan_Id" : plan_choice,
                                "Customer_Id" : cus_id,
                                "Payment_Amount" : amount,
                                "Payment_Time" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj),
                                "Plan_Expiration_Time" : dt.datetime.strptime(date_, "%Y-%m-%d").replace(hour = 23, minute= 59, second = 59) + dt.timedelta(days = duration -1),
                                "Payment_Status": "Completed",
                                "Updated_At" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj)
                }

                payments.append(tuple(payments_dict.values()))

            
                # df_new_payments = pl.DataFrame(payments, schema = payments_schema, orient = "row")
                # df_payments = pl.concat([df_payments, df_new_payments])
                # df_payments.write_csv(data_path_+ payment_file_)
            
            # Generate access
            df_access = pl.read_csv(data_path_+ access_file_, schema=access_schema)

            visit_time = rng.normal(profile_data["hour_mean"], profile_data["hour_std"])

            # if plan_choice  == None:
            #      plan_choice = temp_pay.select(pl.col("Plan_Id")).head(1).item()
            #      print("checando plan_choice", plan_choice)

            access_dict = {"Visit_Id" : access_count + 1,
                    "Customer_Id" : cus_id,
                    "Plan_Id" : plan_choice,
                    "Branch_Id" : 1,
                    "Payment_Id" : str(cus_id) + "-" + str(payment_count),
                    "Date" : dt.datetime.strptime(date_, "%Y-%m-%d").date(),
                    "Visit_Time" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj),
                    "Updated_At" : dt.datetime.combine(dt.datetime.strptime(date_, "%Y-%m-%d"), arrival_time_obj)
                    }

            access_count += 1

            access.append(tuple(access_dict.values()))

    df_new_accesss = pl.DataFrame(access, schema = access_schema, orient = "row")
    df_access = pl.concat([df_access, df_new_accesss])
    df_access.write_csv(data_path_+ access_file_)

    df_new_payments = pl.DataFrame(payments, schema = payments_schema, orient = "row")
    df_payments = pl.concat([df_payments, df_new_payments])
    df_payments.write_csv(data_path_+ payment_file_)

    return df_customers, df_payments, df_access
    

In [12]:
[day.strftime("%Y-%m-%d") for day in pd.date_range(start= "2026-01-02", end = "2026-01-10")]

['2026-01-02',
 '2026-01-03',
 '2026-01-04',
 '2026-01-05',
 '2026-01-06',
 '2026-01-07',
 '2026-01-08',
 '2026-01-09',
 '2026-01-10']

In [13]:
for m_day in [day.strftime("%Y-%m-%d") for day in pd.date_range(start= "2026-02-01", end = "2026-05-30")]:
    datasets = create_customer_access_data(m_day, profiles, gym_hours)

display(datasets[0])
display(datasets[1])
display(datasets[2])

2026-02-01 00:00:00 6 Sunday
68
Total new customers 2
Creating customer
36265
Creating customer
36253
(old customer) Customer 1 with profile 5 appeared False
(old customer) Customer 2 with profile 5 appeared False
(old customer) Customer 3 with profile 1 appeared False
(old customer) Customer 4 with profile 8 appeared False
(old customer) Customer 5 with profile 1 appeared False
(old customer) Customer 6 with profile 5 appeared False
(old customer) Customer 7 with profile 8 appeared False
(old customer) Customer 8 with profile 1 appeared False
(old customer) Customer 9 with profile 1 appeared False
(old customer) Customer 10 with profile 4 appeared False
(old customer) Customer 11 with profile 3 appeared False
(old customer) Customer 12 with profile 3 appeared False
(old customer) Customer 13 with profile 3 appeared False
(old customer) Customer 14 with profile 8 appeared False
(old customer) Customer 15 with profile 2 appeared False
(old customer) Customer 16 with profile 4 appeared F

Customer_Id,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,datetime[μs],f32,f32,i64,str,str,date,str,str,datetime[μs]
1,"""Teresa Irma""","""Laboy Mercado""",false,1989-11-15 00:00:00,21.005507,-101.28949,36253,"""q***********h@gmail.com""","""(473)6602899""",2026-01-02,"""Active""","""5""",2026-05-30 22:32:09.090932
2,"""Claudio José""","""Sáenz Barreto""",true,1978-11-05 00:00:00,20.987411,-101.292892,36255,"""y**********u@hotmail.com""","""(473)3645926""",2026-01-02,"""Active""","""5""",2026-05-30 22:32:12.727524
3,"""Sergio Renato""","""Villagómez Navarrete""",true,2003-11-28 00:00:00,20.962576,-101.280396,36257,"""e**********q@gmail.com""","""(473)5880514""",2026-01-02,"""Active""","""1""",2026-05-30 22:32:16.405117
4,"""Perla Amanda""","""Aguayo Mascareñas""",false,2004-12-22 00:00:00,20.991074,-101.277794,36255,"""i*************p@gmail.com""","""(473)7431576""",2026-01-02,"""Active""","""8""",2026-05-30 22:32:19.791288
5,"""Genaro Guillermo""","""Sáenz Carvajal""",true,1986-05-28 00:00:00,21.005081,-101.295799,36254,"""k**********b@hotmail.com""","""(473)3699298""",2026-01-02,"""Active""","""1""",2026-05-30 22:32:23.484012
…,…,…,…,…,…,…,…,…,…,…,…,…,…
107,"""Jaime Gregorio""","""Durán Ulibarri""",true,2005-11-01 00:00:00,20.99798,-101.278175,36255,"""q**********z@hotmail.com""","""(473)5951336""",2026-05-14,"""Active""","""2""",2026-06-01 21:38:37.156801
108,"""Gerónimo David""","""Gaona Verdugo""",true,1969-07-26 00:00:00,21.000597,-101.249313,36080,"""z****************w@gmail.com""","""(473)8888890""",2026-05-17,"""Active""","""2""",2026-06-01 21:38:40.359627
109,"""Patricio Enrique""","""Henríquez Hernández""",true,1984-11-10 00:00:00,20.981075,-101.288116,36256,"""n****************z@gmail.com""","""(473)4618955""",2026-05-20,"""Active""","""4""",2026-06-01 21:38:43.532746


Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""1-1""",1,1,150.0,2026-01-02 20:10:22,2026-01-02 23:59:59,"""Completed""",2026-01-02 20:10:22
"""2-1""",1,2,150.0,2026-01-02 20:48:54,2026-01-02 23:59:59,"""Completed""",2026-01-02 20:48:54
"""3-1""",1,3,150.0,2026-01-02 06:47:20,2026-01-02 23:59:59,"""Completed""",2026-01-02 06:47:20
"""4-1""",1,4,150.0,2026-01-02 19:23:07,2026-01-02 23:59:59,"""Completed""",2026-01-02 19:23:07
"""5-1""",1,5,150.0,2026-01-02 07:50:41,2026-01-02 23:59:59,"""Completed""",2026-01-02 07:50:41
…,…,…,…,…,…,…,…
"""88-7""",2,88,650.0,2026-05-30 08:34:53,2026-06-28 23:59:59,"""Completed""",2026-05-30 08:34:53
"""89-10""",1,89,150.0,2026-05-30 15:53:02,2026-05-30 23:59:59,"""Completed""",2026-05-30 15:53:02
"""99-8""",1,99,150.0,2026-05-30 07:39:11,2026-05-30 23:59:59,"""Completed""",2026-05-30 07:39:11


Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Visit_Time,Updated_At
i64,i64,i64,i8,str,date,datetime[μs],datetime[μs]
1,1,1,1,"""1-1""",2026-01-02,2026-01-02 20:10:22,2026-01-02 20:10:22
2,2,1,1,"""2-1""",2026-01-02,2026-01-02 20:48:54,2026-01-02 20:48:54
3,3,1,1,"""3-1""",2026-01-02,2026-01-02 06:47:20,2026-01-02 06:47:20
4,4,1,1,"""4-1""",2026-01-02,2026-01-02 19:23:07,2026-01-02 19:23:07
5,5,1,1,"""5-1""",2026-01-02,2026-01-02 07:50:41,2026-01-02 07:50:41
…,…,…,…,…,…,…,…
2557,101,1,1,"""101-4""",2026-05-30,2026-05-30 17:44:45,2026-05-30 17:44:45
2558,102,4,1,"""102-2""",2026-05-30,2026-05-30 07:04:55,2026-05-30 07:04:55
2559,103,4,1,"""103-2""",2026-05-30,2026-05-30 06:18:17,2026-05-30 06:18:17


In [14]:
display(datasets[0])
display(datasets[1])
display(datasets[2])

Customer_Id,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,datetime[μs],f32,f32,i64,str,str,date,str,str,datetime[μs]
1,"""Teresa Irma""","""Laboy Mercado""",false,1989-11-15 00:00:00,21.005507,-101.28949,36253,"""q***********h@gmail.com""","""(473)6602899""",2026-01-02,"""Active""","""5""",2026-05-30 22:32:09.090932
2,"""Claudio José""","""Sáenz Barreto""",true,1978-11-05 00:00:00,20.987411,-101.292892,36255,"""y**********u@hotmail.com""","""(473)3645926""",2026-01-02,"""Active""","""5""",2026-05-30 22:32:12.727524
3,"""Sergio Renato""","""Villagómez Navarrete""",true,2003-11-28 00:00:00,20.962576,-101.280396,36257,"""e**********q@gmail.com""","""(473)5880514""",2026-01-02,"""Active""","""1""",2026-05-30 22:32:16.405117
4,"""Perla Amanda""","""Aguayo Mascareñas""",false,2004-12-22 00:00:00,20.991074,-101.277794,36255,"""i*************p@gmail.com""","""(473)7431576""",2026-01-02,"""Active""","""8""",2026-05-30 22:32:19.791288
5,"""Genaro Guillermo""","""Sáenz Carvajal""",true,1986-05-28 00:00:00,21.005081,-101.295799,36254,"""k**********b@hotmail.com""","""(473)3699298""",2026-01-02,"""Active""","""1""",2026-05-30 22:32:23.484012
…,…,…,…,…,…,…,…,…,…,…,…,…,…
107,"""Jaime Gregorio""","""Durán Ulibarri""",true,2005-11-01 00:00:00,20.99798,-101.278175,36255,"""q**********z@hotmail.com""","""(473)5951336""",2026-05-14,"""Active""","""2""",2026-06-01 21:38:37.156801
108,"""Gerónimo David""","""Gaona Verdugo""",true,1969-07-26 00:00:00,21.000597,-101.249313,36080,"""z****************w@gmail.com""","""(473)8888890""",2026-05-17,"""Active""","""2""",2026-06-01 21:38:40.359627
109,"""Patricio Enrique""","""Henríquez Hernández""",true,1984-11-10 00:00:00,20.981075,-101.288116,36256,"""n****************z@gmail.com""","""(473)4618955""",2026-05-20,"""Active""","""4""",2026-06-01 21:38:43.532746


Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""1-1""",1,1,150.0,2026-01-02 20:10:22,2026-01-02 23:59:59,"""Completed""",2026-01-02 20:10:22
"""2-1""",1,2,150.0,2026-01-02 20:48:54,2026-01-02 23:59:59,"""Completed""",2026-01-02 20:48:54
"""3-1""",1,3,150.0,2026-01-02 06:47:20,2026-01-02 23:59:59,"""Completed""",2026-01-02 06:47:20
"""4-1""",1,4,150.0,2026-01-02 19:23:07,2026-01-02 23:59:59,"""Completed""",2026-01-02 19:23:07
"""5-1""",1,5,150.0,2026-01-02 07:50:41,2026-01-02 23:59:59,"""Completed""",2026-01-02 07:50:41
…,…,…,…,…,…,…,…
"""88-7""",2,88,650.0,2026-05-30 08:34:53,2026-06-28 23:59:59,"""Completed""",2026-05-30 08:34:53
"""89-10""",1,89,150.0,2026-05-30 15:53:02,2026-05-30 23:59:59,"""Completed""",2026-05-30 15:53:02
"""99-8""",1,99,150.0,2026-05-30 07:39:11,2026-05-30 23:59:59,"""Completed""",2026-05-30 07:39:11


Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Visit_Time,Updated_At
i64,i64,i64,i8,str,date,datetime[μs],datetime[μs]
1,1,1,1,"""1-1""",2026-01-02,2026-01-02 20:10:22,2026-01-02 20:10:22
2,2,1,1,"""2-1""",2026-01-02,2026-01-02 20:48:54,2026-01-02 20:48:54
3,3,1,1,"""3-1""",2026-01-02,2026-01-02 06:47:20,2026-01-02 06:47:20
4,4,1,1,"""4-1""",2026-01-02,2026-01-02 19:23:07,2026-01-02 19:23:07
5,5,1,1,"""5-1""",2026-01-02,2026-01-02 07:50:41,2026-01-02 07:50:41
…,…,…,…,…,…,…,…
2557,101,1,1,"""101-4""",2026-05-30,2026-05-30 17:44:45,2026-05-30 17:44:45
2558,102,4,1,"""102-2""",2026-05-30,2026-05-30 07:04:55,2026-05-30 07:04:55
2559,103,4,1,"""103-2""",2026-05-30,2026-05-30 06:18:17,2026-05-30 06:18:17


In [15]:
datasets[1].filter(pl.col("Customer_Id") == 4)

Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""4-1""",1,4,150.0,2026-01-02 19:23:07,2026-01-02 23:59:59,"""Completed""",2026-01-02 19:23:07
"""4-2""",1,4,150.0,2026-01-03 14:23:42,2026-01-03 23:59:59,"""Completed""",2026-01-03 14:23:42
"""4-3""",1,4,150.0,2026-01-05 07:32:18,2026-01-05 23:59:59,"""Completed""",2026-01-05 07:32:18
"""4-4""",1,4,150.0,2026-01-06 18:09:03,2026-01-06 23:59:59,"""Completed""",2026-01-06 18:09:03
"""4-5""",1,4,150.0,2026-01-12 16:42:15,2026-01-12 23:59:59,"""Completed""",2026-01-12 16:42:15
…,…,…,…,…,…,…,…
"""4-20""",1,4,150.0,2026-04-20 11:09:32,2026-04-20 23:59:59,"""Completed""",2026-04-20 11:09:32
"""4-21""",1,4,150.0,2026-05-01 11:01:11,2026-05-01 23:59:59,"""Completed""",2026-05-01 11:01:11
"""4-22""",1,4,150.0,2026-05-09 09:17:35,2026-05-09 23:59:59,"""Completed""",2026-05-09 09:17:35


In [16]:
datasets[2].filter(pl.col("Customer_Id") == 4)

Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Visit_Time,Updated_At
i64,i64,i64,i8,str,date,datetime[μs],datetime[μs]
4,4,1,1,"""4-1""",2026-01-02,2026-01-02 19:23:07,2026-01-02 19:23:07
13,4,1,1,"""4-2""",2026-01-03,2026-01-03 14:23:42,2026-01-03 14:23:42
18,4,1,1,"""4-3""",2026-01-05,2026-01-05 07:32:18,2026-01-05 07:32:18
21,4,1,1,"""4-4""",2026-01-06,2026-01-06 18:09:03,2026-01-06 18:09:03
36,4,1,1,"""4-5""",2026-01-12,2026-01-12 16:42:15,2026-01-12 16:42:15
…,…,…,…,…,…,…,…
2219,4,3,1,"""4-24""",2026-05-08,2026-05-08 09:13:25,2026-05-08 09:13:25
2260,4,3,1,"""4-24""",2026-05-11,2026-05-11 13:00:32,2026-05-11 13:00:32
2270,4,3,1,"""4-24""",2026-05-12,2026-05-12 10:46:17,2026-05-12 10:46:17


In [17]:
datasets[2].group_by("Customer_Id", "Plan_Id").len().sort(by = "len", descending=True)

Customer_Id,Plan_Id,len
i64,i64,u32
29,3,45
11,4,42
18,4,40
9,3,36
48,1,35
…,…,…
91,1,1
49,3,1
98,1,1
